# Purpose: replay experiments for the one control fly to the other replay flies.

In [ ]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq 
from mapd.sinq_builders import build_composite_sinq
import h5py
import glob
import numpy as np
import pickle
import plotly.express as px
import matplotlib.colors as mcolors
# import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import seaborn as sns

import random

%load_ext autoreload
%autoreload 2


%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)


# Fig dir for PCA analysis
fig_dir = './Figure2'

# Load learners vs. control sinq

In [ ]:
sinq_fig1 = Sinq(sinqname='dataset_1')
sinq_ctrl = Sinq(sinqname='dataset_2_controls')
replay_df = sinq_ctrl.display_df().loc[sinq_ctrl.has_tag('replay')]
replay_df

In [ ]:
sinq_fig1.drop_tables()

In [ ]:
T = sinq_fig1.restore_table(dayflycell='250924_F2_C1')
T.extract_trial_properties()
T.fig_folder = fig_dir
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[100:500].index, cmin=-500,cmax=10,format='svg')
fig

In [ ]:
T = sinq_ctrl.restore_table(dayflycell='251205_F1_C1')
T.extract_trial_properties()
T.fig_folder = fig_dir
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')
# fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[300:700].index, cmin=-500,cmax=10,format='svg')
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[100:500].index, cmin=-500,cmax=10,format='svg')


# Figure 2A - probe distributions

In [ ]:
refresh_table_addons()

# --- the only thing to change: probe histogram bin width (pixels) ---
BINWIDTH = 6          # was 2; try 10 or 20
BIN_MIN  = -500
BIN_MAX  = 10          # np.arange(BIN_MIN, BIN_MAX, BINWIDTH) -> keep (BIN_MAX-BIN_MIN) a multiple of BINWIDTH
# --------------------------------------------------------------------

# export the entire distribution
def call_and_export_probe_distributions(row):
    T = sinq_ctrl.restore_table(row.name)
    T.fig_folder = fig_dir
    print('Table: {}'.format(T))
    T.plot_probe_distribution(df_filter=None, index=None, binwidth=BINWIDTH, bin_min=BIN_MIN, bin_max=BIN_MAX, export=True)
    T.plot_probe_distribution(df_filter = {'pyasState':'hi'}, index=None, binwidth=BINWIDTH, bin_min=BIN_MIN, bin_max=BIN_MAX, export=True)
    T.plot_probe_distribution(df_filter = {'pyasState':'lo'}, index=None, binwidth=BINWIDTH, bin_min=BIN_MIN, bin_max=BIN_MAX, export=True)

for dfc in replay_df.index:
    row = sinq_ctrl.df.loc[dfc].copy()
    call_and_export_probe_distributions(row)
    # sinq_ctrl.drop_tables()


## Import and gather histograms

In [ ]:
import glob
import numpy as np
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)  # reset to defaults
mpl.rcParams['pdf.fonttype'] = 42         # embed fonts as text, not paths
mpl.rcParams['svg.fonttype'] = 'none'     # keep text editable in SVG
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 11

import random

folder_path = "./Figure2/exports"
fig_folder = './Figure2'

# 210302_F1_C2 is offset from the others by this much probe position (pixels).
# Expressed in position units so it survives a change of BINWIDTH.
_SHIFT_PX = -20

    # Color scheme for targets, adjust as needed
_force_clrs = [
    (np.float64(0.95447591), np.float64(0.47082238), np.float64(0.32310953)),
    (np.float64(0.7965014), np.float64(0.10506637), np.float64(0.31063031)),
    ]

def ave_and_plot_histograms(dfcs, pat,targets=None,title='None'):
    fig = Figure(figsize=(6,6), dpi=300)
    canvas = FigureCanvas(fig)
    ax = fig.add_subplot(111)

    if targets:
        for key in targets.keys():
            targ_dict = targets[key]
            tgt_clr = _force_clrs[0 if targ_dict['pyasState']=='lo' else 1]
            rect = patches.Rectangle(
                (0, targ_dict['pyasXPosition']),  # (x,y) lower-left corner
                .1,                # width across counts max
                targ_dict['pyasWidth'],            # height
                edgecolor=tgt_clr,
                facecolor=tgt_clr,
                alpha=1
            )
            ax.add_patch(rect)

    normalized_counts_list = []
    probe_bins = None

    for dfc in dfcs:
        pattern=f'{dfc}'+pat
        print(pattern)
        file_path = glob.glob(os.path.join(folder_path, pattern))

        with open(file_path[0], 'rb') as f:
            data = pickle.load(f)
            bins = data['probe_bins']
            
            if probe_bins is None:
                probe_bins = bins
            else:
                assert np.allclose(probe_bins, bins), "Probe bins differ between files!"

            counts = data['total_counts']
            if '210302_F1_C2' in file_path[0]:
                binwidth = bins[1] - bins[0]
                n_shift = int(round(_SHIFT_PX / binwidth))
                print(f'shifting this one fly by {_SHIFT_PX} px = {n_shift} bins (binwidth {binwidth})')
                counts = np.roll(counts, n_shift, axis=0)

            norm_counts = counts / counts.sum()  # Normalize to sum to 1
            normalized_counts_list.append(norm_counts)
            ax.step(norm_counts, probe_bins[:-1], where='post', label=dfc)

    avg_norm_counts = np.mean(normalized_counts_list, axis=0)
    ax.step(avg_norm_counts, probe_bins[:-1], where='post', color='black', linewidth=2, label='Avg Normalized Histogram')

    ax.legend()
    ax.set_xlabel('Probe Position')
    ax.set_ylabel('Normalized Frequency')
    ax.grid(True, alpha=0.5)

    save_path = os.path.join(fig_folder, title + '.svg')
    fig.savefig(save_path,format = 'svg')
    print(f"Saved plot to {save_path}")
    return fig


In [ ]:
targets = sinq_ctrl.restore_table('251205_F1_C1').targets
targets

In [ ]:
dfcs = sinq_fig1.df.index
dfcs

In [ ]:
dfcs

In [ ]:
# dfcs = sinq_fig1.df.loc[sinq_fig1.has_tag('learn')]
figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title= fn+ '_all_learners'))

In [ ]:
replay_df.index

In [ ]:
dfcs = replay_df.index
figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=dfcs,pat=pat,targets=targets,title= fn+'_' +str(BINWIDTH)+'_all_replay'))

In [ ]:
# [SOURCE_HIST] histogram of 250924_F2_C1, the fly whose schedule is replayed, same bins as the replay flies
src_dfc = '250924_F2_C1'
# src_dfc = '250228_F2_C1'
T_src = sinq_fig1.restore_table(src_dfc)
T_src.fig_folder = fig_dir
for df_filter in (None, {'pyasState':'hi'}, {'pyasState':'lo'}):
    T_src.plot_probe_distribution(df_filter=df_filter, index=None, binwidth=BINWIDTH, bin_min=BIN_MIN, bin_max=BIN_MAX, export=True)


In [ ]:
fig, ax, df = T.plot_probe_position_heatmap(index=T.df.loc[300:700].index,
                                            cmin=-500,cmax=10,format='svg')
fig

In [ ]:

figs = []
for pat,fn in zip(['*_probe_dist_lo_*.pkl','*_probe_dist_hi_*.pkl','*_probe_dist_nofilt_*.pkl'],
                  ['probe_dist_lo','probe_dist_hi','probe_dist_nofilt']):
    figs.append(ave_and_plot_histograms(dfcs=[src_dfc],pat=pat,targets=T_src.targets,title= fn+'_' +str(BINWIDTH)+'_'+src_dfc+'_replay_source'))

# 1 sec examples

In [ ]:
T = sinq_ctrl.restore_table(dayflycell='251103_F2_C1')    # 251103_F2_C1
T.extract_trial_properties()
T.fig_folder = fig_dir
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')
fig, ax, df = T.plot_probe_position_heatmap(index=T.df.loc[100:500].index,
                                            cmin=-500,cmax=10,format='svg')
# fig, ax, df = T.plot_probe_position_heatmap(index=T.df.loc[400:800].index,
#                                             cmin=-500,cmax=10,format='svg')
fig

In [ ]:
T = sinq_ctrl.restore_table(dayflycell='251107_F2_C1')
T.extract_trial_properties()
T.fig_folder = fig_dir
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, cmin=-500,cmax=10,format='svg')
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[100:500].index, cmin=-500,cmax=10,format='svg')
fig

In [9]:
# [TRIAL_EXAMPLE] single-trial example: probe position over its target zone, punishment (AS) trace below
# polarity flipped so flexion is positive (probeZero - probe_position)
# uses _force_clrs from the histogram cell for the lo/hi target colours
example_dir = os.path.join(fig_dir, '1sec_examples')
os.makedirs(example_dir, exist_ok=True)

def plot_trial_example(T, trial, fmt='svg', save=True):
    tr = T.df.loc[trial, 'Trial']
    # probe is held constant across each camera frame, so plot one sample per frame (downsample_probe);
    # the full-rate trace put ~10^6 vertices in the svg
    ds = tr.downsample_probe
    t_full = np.asarray(tr.time).ravel()
    t = t_full[ds]
    x = tr.probeZero - np.asarray(tr.probe_position).ravel()[ds]
    # AS is a step signal: keep only its transitions (plus endpoints) so edges stay sample-exact
    ard_full = np.asarray(tr.arduino_output).ravel()[:len(t_full)]
    keep = np.unique(np.r_[0, np.flatnonzero(np.diff(ard_full)) + 1, len(ard_full) - 1])
    t_as, ard = t_full[keep], ard_full[keep]
    tgt_near = tr.probeZero - (tr.pyasXPosition + tr.pyasWidth)   # target edge nearest probeZero
    tgt_clr = _force_clrs[0 if tr.pyasState == 'lo' else 1]

    fig = Figure(figsize=(4, 3.5), dpi=300)
    FigureCanvas(fig)
    ax, ax_as = fig.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.1})

    ax.axhspan(tgt_near, tgt_near + tr.pyasWidth, color=tgt_clr, alpha=0.6, lw=0, label='target')
    ax.axhline(0, color=(0.5, 0.5, 0.5), ls='--', lw=0.75)
    ax.plot(t, x, color='black', lw=1)
    ax.set_ylim(-20, 500)
    ax.set_ylabel('probe position (px)')
    ax.set_title(f'{T.dfc} trial {trial}: {tr.as_outcome} ({tr.pyasState})', fontsize=9)

    ax_as.fill_between(t_as, 0, ard, step='post', color=(0.4, 0.4, 0.4), lw=0)
    ax_as.set_ylim(0, 1.1)
    ax_as.set_yticks([0, 1])
    ax_as.set_ylabel('AS')
    ax_as.set_xlabel('time (s)')
    ax_as.set_xlim(t[0], t[-1])
    for a in (ax, ax_as):
        a.spines[['top', 'right']].set_visible(False)

    if save:
        fn = f'{example_dir}/{T.dfc}_trial_{trial}_example.{fmt}'
        fig.savefig(fn, format=fmt, bbox_inches='tight')
        print('Saved', fn)
    return fig, (ax, ax_as)

In [10]:
T = sinq_ctrl.restore_table(dayflycell='251103_F1_C1')
T.extract_trial_properties()
tr_nums = [131,141,142,157]
for tr_num in tr_nums:
    fig, axs = plot_trial_example(T, tr_num,save=True)

Table 251103_F1_C1 in sinq
Saved ./Figure2\1sec_examples/251103_F1_C1_trial_131_example.svg
Saved ./Figure2\1sec_examples/251103_F1_C1_trial_141_example.svg
Saved ./Figure2\1sec_examples/251103_F1_C1_trial_142_example.svg
Saved ./Figure2\1sec_examples/251103_F1_C1_trial_157_example.svg


# Extension examples

In [ ]:
sinq_ctrl

In [ ]:
T = sinq_ctrl.restore_table(dayflycell='260619_F2_C1')
T.extract_trial_properties()
T.fig_folder = fig_dir
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, extension=True,cmin=-10,cmax=500,format='svg')
fig

In [ ]:
fig, ax, df = T.plot_probe_position_heatmap(index=T.df.loc[300:700].index,
                                            cmin=-10, cmax=500, extension=True, format='svg')
fig

In [ ]:
T = sinq_ctrl.restore_table(dayflycell='260619_F1_C1')
T.extract_trial_properties()
T.fig_folder = fig_dir
fig,ax,df = T.plot_probe_position_heatmap(index=T.df.loc[1:400].index, extension=True,cmin=-10,cmax=500,format='svg')
fig

In [ ]:
fig, ax, df = T.plot_probe_position_heatmap(index=T.df.loc[300:700].index,
                                            cmin=-10, cmax=500, extension=True, format='svg')
fig